# NSG Intervention Profile: Causal Representations vs. Correlation Tracking

This notebook extends the Normalized Simulatability Gain (NSG) framework by testing three variations of the NSG profile (Ordinary, Decorrelated, Targeted). The goal is to determine if models track true causal features or merely rely on correlated proxies, which exposes potential out-of-distribution risks.

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
!pip install -q pandas==2.2.2 numpy==1.26.4

## Imports and Setup

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import pandas as pd
from src.synthetic import make_dataset, decorrelated_counterfactuals
from src.profile import compute_profile
from src.controls import explanation_swap

## Generate Synthetic Correlated Dataset
We build a synthetic dataset with a strict .8$ correlation between 	rue_feature and proxy_feature. This allows us to observe which feature is explicitly relied upon in the model's explanations when generating predictions.

In [ ]:
correlation = 0.8
n = 500
seed = 42

df = make_dataset(n, correlation=correlation, seed=seed)
print(f"Dataset: {len(df)} rows, {len(decorrelated_counterfactuals(df))} decorrelated")

## Evaluate the Three-Variant NSG Profile
1. **Ordinary NSG**: Evaluated across the standard IID distribution.
2. **Decorrelated NSG**: Evaluated strictly on points where true and proxy features diverge.
3. **Targeted NSG**: Checks accuracy against the explanation's *claimed* feature specifically.

We run an **explanation-swap control** (shuffling explanations across rows) to confirm that the NSG gain drops to baseline, verifying that the actual textual content is driving the predictive gain.

In [ ]:
def stub_predictor(row: pd.Series, explanation: str | None) -> int:
    return int(row['true_feature'] > 0)

explanations = [f"The prediction was driven by proxy_feature (value {row['proxy_feature']:.2f})." for _, row in df.iterrows()]

profile = compute_profile(stub_predictor, df, explanations, claimed_feature='proxy_feature')
print("--- Standard Evaluation ---")
print(profile.summary())

swapped_expls = explanation_swap(explanations)
profile_swap = compute_profile(stub_predictor, df, swapped_expls, claimed_feature='proxy_feature')
print("\n--- Explanation-Swap Control (Baseline) ---")
print(profile_swap.summary())